# Black Friday Sales Prediction

A reproducible baseline for predicting `Purchase` from the Black Friday dataset. The notebook is intentionally data-independent at checkout: it does not include generated results, and it stops with an actionable error when the local dataset is unavailable.

## Configuration and reproducibility

Place the Kaggle file at `data/Black Friday Dataset.csv`, or set the `BLACK_FRIDAY_DATA` environment variable to its path before running the notebook. The fixed seed and grouped holdout make the split reproducible and prevent the same `User_ID` from appearing in both train and test sets.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
DEFAULT_DATA_PATH = Path("data") / "Black Friday Dataset.csv"
DATA_PATH = Path(os.environ.get("BLACK_FRIDAY_DATA", DEFAULT_DATA_PATH)).expanduser()

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Black Friday dataset not found. Place 'Black Friday Dataset.csv' "
        "under data/ or set BLACK_FRIDAY_DATA to the CSV path. "
        f"Checked: {DATA_PATH.resolve()}"
    )

In [ ]:
df = pd.read_csv(DATA_PATH)
required_columns = {"Purchase", "User_ID"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")
if not df.columns.is_unique:
    raise ValueError("Dataset must not contain duplicate column names.")
if df.empty:
    raise ValueError("Dataset is empty; provide at least two customer groups and valid rows.")

if not pd.api.types.is_numeric_dtype(df["Purchase"]):
    raise TypeError("Purchase must be numeric for regression.")
if df["Purchase"].isna().any():
    raise ValueError("Purchase contains missing values; remove or impute target rows before modeling.")
if not np.isfinite(df["Purchase"].to_numpy(dtype="float64")).all():
    raise ValueError("Purchase must contain only finite numeric values.")
if df["User_ID"].isna().any():
    raise ValueError("User_ID contains missing values; customer groups must be known for the split.")
if df["User_ID"].nunique(dropna=True) < 2:
    raise ValueError("At least two distinct User_ID groups are required for a train/test split.")

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
print("Columns:", ", ".join(map(str, df.columns)))
display_columns = [c for c in ["Purchase", "User_ID", "Product_ID"] if c in df]
print(df[display_columns].head())


## Basic data checks

The summaries below are descriptive only. Imputation and encoding are fitted later on the training partition inside a scikit-learn pipeline, so test-set information cannot influence preprocessing.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
print(missing[missing > 0])
print("Duplicate rows:", int(df.duplicated().sum()))
print(df["Purchase"].describe())

In [ ]:
import matplotlib.pyplot as plt

ax = df["Purchase"].plot(kind="hist", bins=30, edgecolor="black", title="Purchase distribution")
ax.set_xlabel("Purchase")
plt.tight_layout()
plt.show()

## Leakage-aware split

`User_ID` and `Product_ID` are identifiers rather than stable explanatory variables, so they are excluded from model features. A `GroupShuffleSplit` by `User_ID` is used because the source data contains multiple rows per customer; this gives a more honest estimate for unseen customers than a row-wise random split.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

target = df["Purchase"].copy()
features = df.drop(columns=["Purchase"]).copy()
groups = features.pop("User_ID")
if "Product_ID" in features:
    features = features.drop(columns=["Product_ID"])

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(features, target, groups=groups))
X_train = features.iloc[train_idx].copy()
X_test = features.iloc[test_idx].copy()
y_train = target.iloc[train_idx].copy()
y_test = target.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

if X_train.empty or X_test.empty:
    raise ValueError("The grouped split produced an empty train or test partition.")
if groups_train.nunique() < 1 or groups_test.nunique() < 1:
    raise ValueError("The grouped split must contain at least one customer in each partition.")
assert set(X_train.index).isdisjoint(X_test.index)
assert set(groups_train.unique()).isdisjoint(set(groups_test.unique()))
assert "Purchase" not in X_train.columns
print(f"Train rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Train users: {groups_train.nunique():,}; test users: {groups_test.nunique():,}")


## Preprocessing and models

Numeric missing values use a training-only median; categorical missing values use a training-only most-frequent value; unseen categories are ignored. The encoders and imputers are part of each model pipeline and therefore refit independently inside every cross-validation fold.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool", "string"]
).columns.tolist()
supported_features = set(numeric_features + categorical_features)
unsupported_features = [column for column in X_train.columns if column not in supported_features]
if unsupported_features:
    raise TypeError(
        "Unsupported feature dtypes; convert these columns to numeric or categorical "
        f"values before modeling: {unsupported_features}"
    )
if not supported_features:
    raise ValueError("No usable model features remain after excluding identifier columns.")
all_missing_features = [column for column in X_train.columns if X_train[column].isna().all()]
if all_missing_features:
    raise ValueError(f"Features are entirely missing in the training partition: {all_missing_features}")

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
            ]),
            categorical_features,
        ),
    ],
    remainder="drop",
)

def make_pipeline(model):
    return Pipeline([("preprocess", preprocessor), ("model", model)])

models = {
    "decision_tree": make_pipeline(
        DecisionTreeRegressor(random_state=SEED, min_samples_leaf=2, max_depth=20)
    ),
    "random_forest": make_pipeline(
        RandomForestRegressor(
            n_estimators=100, random_state=SEED, n_jobs=-1, min_samples_leaf=2
        )
    ),
}


In [ ]:
def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    }

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    row = {"model": name, **regression_metrics(y_test, predictions)}
    results.append(row)

results_df = pd.DataFrame(results).sort_values("RMSE")
print(results_df.to_string(index=False))

## Optional grouped cross-validation

The holdout metrics above are the default lightweight check. Set `RUN_CV = True` to run a three-fold grouped validation; this can be expensive for the random forest on the full Kaggle dataset.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_validate

RUN_CV = False
if RUN_CV:
    if groups.nunique() < 3:
        raise ValueError("RUN_CV=True requires at least three distinct User_ID groups.")
    cv = GroupKFold(n_splits=3)
    scoring = {
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    }
    cv_result = cross_validate(
        models["random_forest"],
        features,
        target,
        groups=groups,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
    )
    print({
        "MAE": -cv_result["test_mae"].mean(),
        "RMSE": -cv_result["test_rmse"].mean(),
        "R2": cv_result["test_r2"].mean(),
    })
else:
    print("RUN_CV=False; skipped optional grouped cross-validation.")


## Interpretation and limitations

- No model score is reported in the repository because the dataset is not included.
- Grouped splitting estimates performance for unseen customers; it may differ from a row-wise split used in some tutorials.
- The default models are baselines, not tuned production models. Compare MAE, RMSE, and R² together, and inspect residuals before deployment.
- Do not commit the Kaggle data file or derived artifacts to this repository.